# 確率分布

## 導入

この Notebook では，SciPy の統計モジュール `scipy.stats` を用いて
種々の確率分布の性質や形（グラフ）を見ていく。

この Notebook のオリジナルは GitHub から入手できる。

https://github.com/sasaki-shigeo/Jupyter/blob/main/Distribution.ipynb

また Google Colaboratory にて Notebook の実行が可能である。

https://colab.research.google.com/github/sasaki-shigeo/Jupyter/blob/main/Distribution.ipynb

### 前提知識
次の事項は既知とする。

* 確率分布の基本概念
    * 確率変数と確率分布
    * 期待値と分散
    * 確率密度関数 (PDF), 累積分布関数 (CDF)
* 代表的な確率分布
    * 二項分布
    * 正規分布

二項分布や正規分布を知っている前提で，
期待値などの統計量を求める関数の説明をするし，
統計量や分布関数を知っている前提で，様々な確率分布の説明をする。

### 使用する Python モジュール

この Notebook では
`matplotlib`, `numpy`, `pandas`, `scipy` といった
Python モジュールを必要とする。
Jupyter Notebook が動作する環境においてはインストール済みのはずだが，
次のセルを実行すれば，インストールされているか（import 可能か）確認できる。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

もし import に関するエラーが発生するなら
モジュールのインストール等の対処が必要である。
インストール方法は，Jupyter Notebook の動作環境によって異なるので，
確認の上，作業してもらいたい。

### スニペットとしての利用

この Notebook では，コード・セルをスニペットとして利用できるようにするため，
原則，import を省略せずに記載している。
スニペットとは，コピペして使う，典型的用例となる小プログラムのことである。
この Notebook の多くのコード・セルは，途中のセルをスキップしても実行可能である。

ただし，一部のコードセルは，
前のセルで作られた変数を使用しているなどの理由で，単独実行できない。
モジュールが不足している旨のエラーメッセージが表示されるときは，
このノートブックを先頭から実行するか，
セクションの先頭から実行し直すなどしてもらいたい。

## SciPy の統計モジュール

### 主な離散確率分布

`scipy.stats` が扱うことのできる主な**離散**確率分布を挙げる。

| 確率分布            | 摘要 |
|--------------------|-----|
| randint($a$, $b$)  | 区間 $a\leqq X < b$ の離散一様分布|
| binom($n$, $p$)    | 試行数 $n$, 成功確率 $p$ の二項分布 |
| bernoulli($p$)     | 成功確率 $p$ ベルヌーイ分布（試行数1の二項分布）|
| poisson($\mu$)     | 平均値 $\mu$ のポアソン分布 |
| geom($p$)          | 成功確率 $p$ の幾何分布（成功するまでの試行回数）|
| hypergoem($M$, $n$, $N$) | 全体で $M$ 個のうち$n$個の当たりがあって$N$回選ぶ超幾何分布 |
| nbinom($n$, $p$)   | 成功確率 $p$ で $n$ 回成功するまでの失敗回数（負の二項分布）|
| zipf($\alpha$)     | ゼータ関数 $\zeta(\alpha)$ による Zipf分布 |

### 主な連続確率分布

こちらは `scipy.stats` が提供する主な**連続**確率分布である。

| 確率分布          | 摘要 |
|-----------------------|-----------|
| uniform(loc=$a$, scale=$b$)      | $a \leqq X < a+b$ の一様分布（デフォルト $a=0, b=1$） |
| triang($c$, loc=$a$, scale=$b$)  | $a \leqq X < a+b$ で頂点 $X=a+bc$ の三角分布（デフォルト $a=0, b=1$） |
| norm(loc=$\mu$, scale=$\sigma$)  | 平均値 $\mu=0$, 標準偏差 $\sigma=1$ の正規分布（デフォルト $\mu=0, \sigma=1$） |
| chi2($\nu$)           | 自由度 $\nu$ のカイ2乗分布 |
| t($\nu$)              | 自由度 $\nu$ のスチューデント t 分布 |
| f($\nu_1$, $\nu_2$)   | 自由度 $\nu_1$, $\nu_2$ の F分布 |
| cauchy()              | コーシー分布（自由度 1の t分布に同じ）|
| expon(scale=$1/\lambda$)  | 指数分布（デフォルト $\lambda = 1$）|
| halfnorm()            | 正規分布の右半分 |
| lognorm()   　　　　　  | 対数正規分布 |
| laplace()             | ラプラス分布 |
| pareto($b$)           | パレート分布 |

### 確率分布に関する関数

`scipy.stats` は確率分布ごとに様々な関数を提供している。

| メソッド        | 摘要 |
|----------------|-----|
| mean()         | 期待値（平均値） |
| median()       | 中央値 |
| var()          | 分散 |
| std()          | 標準偏差 |
| stats()        | 期待値，分散，歪度，尖度 |
| pmf($x$)       | 確率質量関数 |
| pdf($x$)       | 確率密度関数 |
| cdf($x$)       | 累積分布関数 $\int_{-\infty}^x \mathrm{pdf}(t)\,dt$|
| sf($x$)        | 生存関数 $1-\mathrm{cdf}(x) = \int_x^{\infty} \mathrm{pdf}(t)\,dt$ |
| ppf($p$)       | パーセント点（パーセンタイルの確率分布版，累積分布の逆関数）|
| isf($p$)       | 上側パーセント点 ppf($1-p$), 生存関数の逆関数 (inverse of sf) |
| rvs(size=$n$)  | 確率分布に従う乱数列 |

`mean`, `median`, `var`, `std` といった関数により確率分布の統計量を計算できる。

In [ ]:
from scipy.stats import binom         # 二項分布 binom を import

dist = binom(n = 6, p = 1/2)          # 試行数 n=6, 成功確率 p = 1/2 の二項分布

dist.mean()

歪度 (skewness), 尖度 (kurtosis) は
3次モーメント以上の統計量については，

確率密度関数 (Probability Density Function) `pdf`,
累積分布関数 (cumulative distribution function) `cdf`,
生存関数 (survival function) `sf` は，
確率分布を定める関数である。

離散確率分布では，確率密度関数 `pdf` の代わりに
確率質量関数 (probability mass function) `pmf` を使う。
$\mathrm{pmf}(x)$ は確率変数 $X$ の実現値が $x$ のときの確率
$\Pr(X = x)$ のことである。つまり

$$
\mathrm{pmf}(x) = \Pr(X = x)
$$

一方，連続確率分布では，
確率変数 $X$ が特定の値 $x$ をとる確率はほとんど 0に等しい。つまり

$$
\Pr(X = x) = 0
$$

なので確率質量関数は無意味である。
代わりに累積分布関数 `cdf` を微分した関数である
確率密度関数 `pdf` を使う。

$$
\begin{align}
\mathrm{cdf}(x) &= \Pr(X < x) \\
\mathrm{pdf}(x) &= \mathrm{cdf}'(x)
\end{align}
$$

確率統計の教科書を読むと，確率分布を確率密度関数で定義することが多いが，
これらの関数は相互に変換できるので，どれかが優越しているということはない。
また確率分布を定義する別の方法として，
ある確率分布に従う確率変数を加算したり変換した結果を
新たな確率変数とする分布を考えることもある。

たとえば，平均値 $\mu=0$, 分散 $\sigma^2=1$ の標準正規分布に
独立に従う確率変数 $X_1, X_2, \dots, X_n$ に対し，
確率変数 $Y = X_1^2 + X_2^2 + \dots + X_n^2$ が従う確率分布は
カイ2乗分布と定義される。

また確率変数 $\Theta$ が区間 $-\pi/4 < \Theta < \pi/4$ の一様分布に従うとき，
確率変数 $Y = \tan(\Theta)$ が従う確率分布はコーシー分布と定義される。

これらの定義から確率密度関数の式を導出できるが，
逆に，その式から確率分布を定義する教科書もある。
この辺は柔軟に理解してもらいたい。

関数 `rvs` (random variates) は，
確率分布に従う乱数列を生成する関数である。
random variate = random variable とは確率変数のことである。
確率変数の列 (sequence) と理解しておけばよいだろう。
サイズ $n$ の無作為抽出された標本 (random sample) とみなすとよいこともある。

### 確率分布のパラメータの指定方法

確率分布はパラメータを決めることで一意に定まる。
たとえば二項分布 $B(n, p)$ では，試行数 $n$, 成功確率 $p$ がパラメータである。
正規分布 $N(\mu, \sigma^2)$ では平均値 $\mu$, 分散 $\sigma^2$ がパラメータである。

さて確率論の分野ではパラメータを母数と訳すが，
この訳語は，分母や母集団の語に引きずられて誤解を招きやすい。
むしろ関数やサブルーチンの引数 (parameter, argument) に近いので，
このノートブックではパラメータの語を用いる。

話題を戻すと `scipy.stats` における確率分布にまつわる関数を使うには，
確率分布のパラメータを与えなければならない。
`scipy.stats` では次のどちらかの方法で与える。

1. まずコンストラクタに分布のパラメータを与えて確率分布オブジェクトを作る。
   確率分布にまつわる関数はこのオブジェクトのインスタンスメソッドとして提供される。
2. 確率分布モジュール直下に提供される関数を使う。
   分布のパラメータはこの関数に直接与え，確率分布オブジェクトは作らない。

方法1で，試行数 $n = 4$, 成功確率 $p = 1/2$ の
二項分布 $B(4, 1/2)$ の確率分布オブジェクト dist を作り，
その確率質量関数 pmf の表を得るには次のようにする。

In [ ]:
from scipy.stats import binom     # 二項分布 binom を import
import pandas as pd

n = 4
p = 1/2
dist = binom(n, p)
xs = range(n + 1)
ys = dist.pmf(xs)

pd.DataFrame({'確率': ys})

コンストラクタ binom のパラメータ $n$, $p$ は
キーワード引数として与えることが可能である。
キーワード引数は任意の順に記載して構わない。

In [ ]:
from scipy.stats import binom
import pandas as pd

dist = binom(p = 1/2, n = 4)      # 引数名を明示すれば順番は自由
xs = range(n + 1)
ys = dist.pmf(xs)

pd.DataFrame({'確率': ys})

関数 pmf に確率分布のパラメータを与えれば
確率分布オブジェクトを作らなくてもよい。
次のように書いて同じ結果が得られる。

In [ ]:
from scipy.stats import binom     # 二項分布 binom を import
import pandas as pd

xs = range(n + 1)
ys = binom.pmf(xs, n = 4, p = 1/2)

pd.DataFrame({'確率': ys})

たとえば
標準正規分布（平均値 0.0, 標準偏差 1.0 の正規分布）の確率分布オブジェクト dist を作り，その確率密度関数 (PDF) のグラフを描くには次のようにする。

In [ ]:
import numpy as np
from scipy.stats import norm      # norm (正規分布) を import
import matplotlib.pyplot as plt

std_norm = norm()                 # 標準正規分布
xs = np.linspace(-4, 4, 100)      # -4以上 4以下を 100分割する内分点の配列
ys = std_norm.pdf(xs)             # 確率密度関数
plt.plot(xs, ys)

一方，次のようにすれば，確率分布オブジェクトを作らず，
標準正規分布の確率密度関数 (PDF) をプロットできる。

In [ ]:
import numpy as np
from scipy.stats import norm
import matplotlib.pyplot as plt

xs = np.linspace(-4, 4, 100)
ys = norm.pdf(xs)                 # 標準正規分布の確率密度関数
plt.plot(xs, ys)

正規分布 `norm` は，
位置パラメータ `loc` で平均値 $\mu$ を
スケールパラメータ `scale` で標準偏差 $\sigma$ を指定できる。

In [ ]:
import numpy as np
from scipy.stats import norm
import matplotlib.pyplot as plt

xs = np.linspace(-5, 5, 100)
ys = norm.pdf(xs, loc = 1.0, scale = 1.5)
plt.plot(xs, ys, label = "μ=1, σ=1.5")
zs = norm.pdf(xs, loc = 0.0, scale = 1.0)
plt.plot(xs, zs, label = "μ=0, σ=1.0")
plt.legend()

なお `scipy.stats` の多くの連続確率分布も
位置パラメータ `loc` とスケールパラメータ `scale` を持つ。
それぞれの意味するところは確率分布によって異なるので，
正確なところは `scipy.stats` のドキュメントを参照のこと。

### 関数のブロードキャストについて

`plot` メソッドには x座標の列と y座標の列を与える。
関数 $y = f(x)$ のグラフを描くには
x座標の列 $[x_1, x_2, \ldots, x_n]$ を用意しておき，
そこから y座標の列 $[y_1, y_2, \ldots, y_n]$ を作って
`plot` メソッドに与えればよい。
このときブロードキャストを使うのが望ましい。

ここでいうブロードキャストとは，
関数 $y = f(x)$ に対してベクトル（リストや配列や行列）を与える，
たとえばリスト $[x_1, x_2, \ldots, x_n]$ を与えると
$[f(x_1), f(x_2), \ldots, f(x_n)]$ を返す仕組みのことである。

たとえば `numpy` モジュールの `sqrt` 関数はブロードキャストに対応しており，
次のような計算ができる。

In [ ]:
from numpy import sqrt

sqrt([1, 4, 9, 16, 25])

一方 `math` モジュールの `sqrt` 関数は
ブロードキャストに対応しない。
もし同じような結果を得たいなら，
たとえば内包表記を使って次のように書く必要がある。

In [ ]:
from math import sqrt

[ sqrt(x) for x in [1, 4, 9, 16, 25] ]

両者はデータ型が異なるが，
`matplotlib` の `plot` メソッドは，どちらもデータ列として扱い，
各点を結ぶグラフを描く。

このノートブックでは，
ベクトルの変数を xs, ys のように，変数名の末尾に s を付けて，
スカラーの変数 x, y と区別するようにしている。
プログラムの<ruby>字面<rp> (</rp><rt>じづら</rt><rp>) </rp></ruby>を見るだけで
スカラーに対する処理をしているのか，
ベクトル（配列）に対する処理をしているのか，
瞬時に区別できるようにするためである。

### 平均，分散など

`mean` 関数で平均値，
`var` 関数で分散 (variance)，
`std` 関数で標準偏差 (standart deviation) が得られる。

In [ ]:
from scipy.stats import binom

dist = binom(n = 10, p = 1/2)
print("平均値", dist.mean())
print("分散", dist.var())
print("標準偏差", dist.std())

`binom.stats` 関数で，二項分布の平均値，分散が得られる。
関数のキーワードパラメータとして `moments='mvsk'` も与えると，
平均値 (mean), 分散 (variance) だけでなく，
<ruby>歪度<rp>(</rp><rt>わいど</rt><rp>)</rp></ruby> (skewness),
<ruby>尖度<rp>(</rp><rt>せんど</rt><rp>)</rp></ruby> (kurtosis) が得られる。

In [ ]:
# 変数 dist は前のセルで定義済み

m, v = dist.stats()
print(m, v)

m, v, s, k = dist.stats(moments='mvsk')
print(m, v, s, k)

## 二項分布　(binomial distribution)

### 二項分布の確率質量関数

`scipy.stats` の `binom` により二項分布を扱える。

次のセルにより，
試行回数 $n = 2$, 成功確率 $p = 1/2$ の二項分布 $B(2, 1/2)$ の確率質量関数から
成功回数 $X = k$ と確率 $\Pr(X = k)$ の表を得る。

In [ ]:
from scipy.stats import binom
import pandas as pd

n = 2
p = 1/2
xs = range(n+1)
ys = binom.pmf(xs, n, p)

pd.DataFrame({'確率': ys})

次は $n = 3$, $p = 1/2$ の二項分布

In [ ]:
from scipy.stats import binom
import pandas as pd

n = 3
p = 1/2
xs = range(n+1)
ys = binom.pmf(xs, n, p)

pd.DataFrame({'確率': ys})

次は $n = 4$, $p = 1/2$ の二項分布

In [ ]:
from scipy.stats import binom
import pandas as pd

n = 4
p = 1/2
xs = range(n+1)
ys = binom.pmf(xs, n, p)
pd.DataFrame({'確率': ys})

グラフを描いてみる。

In [ ]:
from scipy.stats import binom
import matplotlib.pyplot as plt

n = 4
p = 1/2
xs = range(n+1)
ys = binom.pmf(xs, n, p)
plt.bar(xs, ys)

成功確率が 1/2 より小さい ($p < 1/2$) と左に偏る。

In [ ]:
from scipy.stats import binom
import pandas as pd
import matplotlib.pyplot as plt

n = 6
p = 1/3
xs = range(n+1)
ys = binom.pmf(xs, n, p)
plt.bar(xs, ys)

成功確率 $p > 1/2$ だと右に偏る。

In [ ]:
from scipy.stats import binom
import pandas as pd
import matplotlib.pyplot as plt

n = 6
p = 2/3
xs = range(n+1)
ys = binom.pmf(xs, n, p)
plt.bar(xs, ys)

### 期待値など

証明は示さないが，二項分布の期待値および分散は次のようになる。

確率変数 $X$ が，試行数 $n$, 成功確率 $p$ の二項分布 $B(n, p)$  に従うとする。
また失敗確率を $q = 1 - p$ で表すと，
$X$ の**期待値**は
$$
E(X) = np
$$

$X$ の**分散**は
$$
V(X) = npq
$$
である。

## 正規分布

`scipy.stats.norm` により正規分布の処理ができる。

### 分布の形

In [ ]:
from scipy.stats import norm
import numpy as np
import matplotlib.pyplot as plt

xs = np.linspace(-4, 4, 100)
ys = norm.pdf(xs)
plt.ylim(0, 0.47)
spines = plt.gca().spines
spines['left'].set_position('zero')
spines['right'].set_visible(False)
spines['top'].set_visible(False)
plt.plot(xs, ys)

キーワード引数 loc により平均値を指定できる。
loc は，location parameter の意。

平均値 $\mu$ が変わるとグラフは平行移動する。

In [ ]:
from scipy.stats import norm
import numpy as np
import matplotlib.pyplot as plt

xs = np.linspace(-4, 4, 100)

plt.plot(xs, norm.pdf(xs, loc=-1.0), label = "μ=-1")
plt.plot(xs, norm.pdf(xs, loc=0.0),  label = "μ=0")
plt.plot(xs, norm.pdf(xs, loc=1.0),  label = "μ=1")
plt.ylim(0, 0.47)
spines = plt.gca().spines
spines['left'].set_position('zero')
spines['right'].set_visible(False)
spines['top'].set_visible(False)
plt.legend()

キーワード引数 scale により**標準偏差**を指定できる。
確率統計の教科書では，正規分布のパラメータは分散 $\sigma^2$ だが，
多くのソフトウェアでは標準偏差 $\sigma$ を指定する。

&sigma; が大きくなると**ばらつき**が大きくなり，峰 (peak) が低くなる。

In [ ]:
xs = np.linspace(-4, 4, 100)
plt.plot(xs, norm.pdf(xs), label="σ=1.0")
plt.plot(xs, norm.pdf(xs, scale=1.4), label="σ=1.4") 
plt.legend()

&sigma; が小さいと**ばらつき**が小さくなり，峰 (peak) が高くなる。

In [ ]:
xs = np.linspace(-4, 4, 100)
plt.plot(xs, norm.pdf(xs), label="σ=1.0")
plt.plot(xs, norm.pdf(xs, scale=0.7), label="σ=0.7")
plt.legend()

## カイ2乗分布

### 定義

このセクションは，
(1)カイ2乗分布が天下り式に定義されるのが気持ち悪いという人，
(2)カイ2乗分布がバラバラな用途に使われるが何故だろうと思った人のための説明である。
カイ2乗分布が分散と密接に関わることが見て取れればよい。

> 確率変数 $X_1, X_2, \dots, X_n$ が
独立に標準正規分布 $N(\mu=0, \sigma^2=1)$ に従うとき，平方の和
$$
X_1^2 + X_2^2 + \dots + X_n^2
$$
が従う確率分布を**自由度 $n$ のカイ2乗分布**といい $\chi^2(n)$ と表す。

これを $n$ で割れば分散に関する分布が得られそうだ。
あまりポピュラーではないが，これには修正カイ2乗分布という名前が付いている。

> 確率変数 $X$ が自由度 $n$ のカイ2乗分布に従う ($X\approx\chi^2_n$) とき，
$X/n$ が従う確率分布を修正カイ2乗分布といい $C^2_n$ と表す。すなわち
$$
X\approx\chi^2_n \implies X/n \approx C^2(n)
$$

後述する $t$ 分布や $F$ 分布を定義するときは修正カイ2乗分布を使った方が簡単である。
といっても，これだけのことなので大きな差はない。

### カイ2乘分布と自由度

サイズ $n$ の標本 $\{X_1, X_2, \dots, X_n\}$ に対する推定や検定では，
自由度 $n-1$ のカイ2乘分布や自由度 $n-1$ の _t_ 分布を用いる。
自由度が 1つ減る理由に関わる定理を取り上げる。

まず次の命題を考える。カイ2乗分布の定義を言い換えただけである。
> 確率変数 $Z_1, Z_2, \dots, Z_n$ が
独立に標準正規分布 $N(\mu=0, \sigma^2=1)$ に従うとき
$$
\sum_{i=1,\ldots,n} Z_i^2 = Z_1^2 + Z_2^2 + \dots + Z_n^2
$$
は自由度 $n$ のカイ2乗分布 $\chi^2(n)$ に従う。

正規分布の標準化変換
$$
Z = \frac{X-\mu}{\sigma}
$$
を逆向きに適用すると，一般の正規分布の命題になる。

> 確率変数 $X_1, X_2, \dots, X_n$ が
独立に正規分布 $N(\mu, \sigma^2)$ に従うとき
$$
\sum_{i=1,\dots,n} \left(\frac{X_i-\mu}{\sigma}\right)^2
$$
は自由度 $n$ のカイ2乗分布 $\chi^2(n)$ に従う。

この命題の $\mu$（母平均）を $\overline{X}$（標本平均）に変えると自由度が1つ減る。

> 確率変数 $X_1, X_2, \dots, X_n$ が
独立に正規分布 $N(\mu, \sigma^2)$ に従うとき
$$
\sum_{i=1,\dots,n} \left(\frac{X_i-\overline{X}}{\sigma}\right)^2
$$
は自由度 $n-1$ のカイ2乗分布 $\chi^2(n-1)$ に従う。

この命題の意味するところは，左辺には見かけ上 $n$ 個の変数があるが，
実質的な変数（独立な変数）は $n-1$ 個しかないということである。

一般の $n$ の場合の証明は大変なので$n=2$ の場合だけ確かめる。
標準化変換してしまえばよいので $Z_1, Z_2$ が独立に
標準正規分布 $N(\mu=0, \sigma^2=1)$ に従う場合だけ考える。

このとき
$$
\begin{align}
\overline{Z} &=(Z_1+Z_2)/2 \\
Z_1-\overline{Z} &= (Z_1-Z_2)/2 \\
Z_2-\overline{Z} &= (Z_2-Z_1)/2
\end{align}
$$

$Z_1-\overline{Z}$ と $Z_2-\overline{Z}$ は符号が逆なだけであり
1個の変数に従属する。
適切な大きさの変数を見つけてやれば，自由度が1であることを示せるはずだ。

まず，正規分布の再生性から $Z_1+Z_2$ は正規分布に従う。
$Z_1, Z_2\approx N(\mu=0, \sigma^2=1)$ の和の平均値が 0なのは直ちにわかる。

$$
E(Z_1+Z_2) = E(Z_1)+E(Z_2) = 0
$$

和の分散は
$$
V(Z_1+Z_2) = V(Z_1)+V(Z_2) = 1 + 1 = 2
$$

よって
$$
Z_1+Z_2\approx N(\mu=0, \sigma^2=2)
$$

同じ理由で
$$
Z_1-Z_2 \approx N(\mu=0, \sigma^2=2)
$$

また $V(a X) = a^2 V(X)$ であることから

$$
(Z_1-Z_2)/\sqrt{2} \approx N(\mu=0, \sigma^2=1)
$$

$Y=(Z_1-Z_2)/\sqrt{2}$ として，元の命題が正しいことを確かめる。

$$
\begin{align}
(Z_1-\overline{Z})^2 + (Z_2-\overline{Z})^2
&= \left(\frac{Y}{\sqrt{2}}\right)^2 + \left(-\frac{Y}{\sqrt{2}}\right)^2 \\
&= \frac{Y^2}{2} + \frac{Y^2}{2} \\
&= Y^2 \\ \approx \chi^2(1)
\end{align}
$$

となり，自由度1のカイ2乗分布に従うことがわかる。

$n \geq 3$ の場合，具体的な _Y_ たちを見つけるには
線形代数（と数式処理ソフト）の助けを借りる必要がある。

$n = 3$ でいうと，
まず $Z_1, Z_2, Z_3$ から
$Z_1-\overline{Z}, Z_2-\overline{Z}, Z_3-\overline{Z}$ を作る行列を考える。

$$
\begin{pmatrix}
Z_1-\overline{Z} \\
Z_2-\overline{Z} \\
Z_3-\overline{Z}
\end{pmatrix} =
\begin{pmatrix}
 2/3 & -1/3 & -1/3 \\
-1/3 &  2/3 & -1/3 \\
-1/3 & -1/3 &  2/3
\end{pmatrix}
\begin{pmatrix}
Z_1 \\ Z_2 \\ Z_3
\end{pmatrix}
$$
この式に現れる行列を _A_ とする。
_A_ は偏差の計算を表している。
これを正規直交化し，そこから _Y_ たちを得る。

_A_ の固有値を求めると 1, 1, 0 である。
実際，_A_ は正則行列ではない。
$$
\begin{gather}
\det A = 0 \\
\mathrm{rank} A = 2
\end{gather}
$$

固有値 0に対応する固有ベクトルを $\displaystyle\frac{1}{\sqrt{3}}(1, 1, 1)$
とする。
他の固有ベクトルを求めるのは，手計算では大変なので，
数式処理システムの力を借りた (Mathematica で正規直交化した）が，
$$
\begin{gather}
(\sqrt{2/3}, -1\sqrt{6}, -1/\sqrt{6}) \\
(0, 1/\sqrt{2}, -1/\sqrt{2})
\end{gather}
$$
が得られた。
こうして得られる直交行列によって

$$
\begin{pmatrix}
Y_1 \\ Y_2 \\ Y_3
\end{pmatrix} =
\begin{pmatrix}
 1/\sqrt{3} & 1/\sqrt{3} & 1/\sqrt{3} \\
 \sqrt{2/3} & -1/\sqrt{6} & -1/\sqrt{6} \\
 0          & 1/\sqrt{2} &  1/\sqrt{2}
\end{pmatrix}
\begin{pmatrix}
Z_1 \\ Z_2 \\ Z_3
\end{pmatrix}
$$

とする。正規分布の再生性により $Y_1, Y_2, Y_3$ は正規分布である。
平均値が 0なのは直ちにわかる。
分散は
$$
\begin{align}
V(Y_1) &= \frac{1}{3}V(Z_1) + \frac{1}{3}V(Z_2) + \frac{1}{3}V(Z_3) = 1 \\
V(Y_2) &= \frac{2}{3}V(Z_1) + \frac{1}{6}V(Z_2) + \frac{1}{6}V(Z_3) = 1 \\
V(Y_3) &= \phantom{\frac{2}{3}V(Z_1)+} 
                              \frac{1}{2}V(V_2) + \frac{1}{2}V(Z_3) = 1
\end{align}
$$
したがって $Y_1, Y_2, Y_3$ はいずれも標準正規分布に従う。
このことから

$$
Z_1^2 + Z_2^2 + Z_3^2 = Y_1^2 + Y_2^2 + Y_3^2
$$

ここで
$$
Y_1^2 = \left(\frac{Z_1 + Z_2 + Z_3}{\sqrt{3}}\right)^2
      = (\sqrt{3}\cdot\overline{Z})^2
      = 3\overline{Z}^2
$$

なので
$$
\begin{align}
\sum_{i=1,2,3} (Z_i - \overline{Z})^2 \\
&= \sum_{i=1,2,3} Z_i^2 - 3\overline{Z}^2 \\
&= \sum_{i=1,2,3} Z_i^2 - Y_1^2 \\
&= Y_2^2 + Y_3^2
&\approx \chi^2(2)
\end{align}
$$

自由度が 2であることがわかった。

$n \geq 4$ でも同じようにできる。
要所は次の3つである。

1. 標本 $Z_1, Z_2, \dots, Z_n$ から
   標本平均からの偏差 $(Z_1 - \overline{Z}), (Z_2-\overline{Z}), \dots, (Z_n-\overline{Z})$ を得る計算を線形代数の観点から見ると，
   ランクが 1落ちる変換である。
   つまり変数が 1つ減る。自由度が1つ減るのはこのせいである。
2. 固有値は $1, 1, \dots, 1, 0$ になる。固有値 0に対応する固有ベクトルを
   $(1/\sqrt{n}, 1/\sqrt{n}, \dots, 1 /\sqrt{n})$ にする。
   これは平均値 $\overline{Z}$ に相当するベクトルを長さ1にしたものである。
4. カイ2乗分布の定義に適合する変数を見つけるのは大変だが，
   正規直交化して固有ベクトルを得ればよい。
   直交行列が存在することが重要で，具体的な式はわからなくて構わない。


### 期待値など

$X\approx\chi^2(n)$ とする。

* 平均値 $E(X) = n$
* 分散 $V(X) = 2n$
* 歪度 $\sqrt{n/8}$
* 尖度 $12/n$

### 自由度 1

In [ ]:
from scipy.stats import chi2
import matplotlib.pyplot as plt
import numpy as np

xs = np.linspace(0, 5, 100)
plt.plot(xs, chi2.pdf(xs, df=1))

生存関数

$X=6$ くらいで片側有意水準 5% になる。

In [ ]:
from scipy.stats import chi2
import matplotlib.pyplot as plt
import numpy as np

xs = np.linspace(0, 6, 100)
plt.plot(xs, chi2.sf(xs, df=1))

より直接的に，
確率から _X_ を得るにはパーセント点（累積分布関数の逆関数）を使う。

95%（有意水準5%）で $X=6$ くらいになる。

In [ ]:
from scipy.stats import chi2
import matplotlib.pyplot as plt
import numpy as np

ps = np.linspace(0, 1, 100)
plt.plot(ps, chi2.ppf(ps, df=1))
ax = plt.gca()
ax.xaxis.set_major_formatter(ticker.PercentFormatter(1))

### 自由度 2

自由度2のカイ2乗分布は，$\lambda = 1/2$ （平均値 $1/\lambda = 2$）の
指数分布と一致する。

確率密度関数

In [ ]:
from scipy.stats import chi2
import matplotlib.pyplot as plt
import numpy as np

xs = np.linspace(0, 5, 100)
plt.plot(xs, chi2.pdf(xs, df=2))

生存関数関数 $1-\mathrm{cdf}(X)$

In [ ]:
from scipy.stats import chi2
import matplotlib.pyplot as plt
from matplotlib import ticker
import numpy as np

xs = np.linspace(0, 5, 100)
plt.plot(xs, chi2.sf(xs, df=2))
ax = plt.gca()
ax.yaxis.set_major_formatter(ticker.PercentFormatter(1))

パーセント点（累積分布関数の逆関数）

In [ ]:
from scipy.stats import chi2
import matplotlib.pyplot as plt
import numpy as np

ps = np.linspace(0, 1, 100)
plt.plot(ps, chi2.ppf(ps, df=2))
ax = plt.gca()
ax.xaxis.set_major_formatter(ticker.PercentFormatter(1))

## t分布

&infin;

In [ ]:
from scipy.stats import t, norm
import matplotlib.pyplot as plt
import numpy as np

xs = np.linspace(-4, 4, 100)
plt.plot(xs, norm.pdf(xs), label='df=∞')
plt.plot(xs, t.pdf(xs, df=1), label='df=1')
plt.plot(xs, t.pdf(xs, df=2), label='df=2')
plt.plot(xs, t.pdf(xs, df=5), label='df=5')
plt.legend()

## ベルヌーイ分布とダミー変数

結果が成功と失敗のどちらかである試行をベルヌーイ試行という。
要するに，二項分布の1回の試行のことである。
試行数 1の二項分布 $B(1, p)$ のことをベルヌーイ分布と呼ぶこともある。
わざわざ名前を付けるほどのものでもなさそうだが，
その性質は，統計処理におけるダミー変数と関連がある。

### ベルヌーイ分布の期待値と分散
ベルヌーイ試行の成功確率を $p$, 失敗確率を $q = 1 - p$ とする。
期待値と分散は，二項分布 $B(1, p)$ のものと同じなので

$$
\begin{align}
E(X) &= p \\
V(X) &= pq
\end{align}
$$

である。難しくないので $V(X)$ の導出を示す。
$\bar{x} = E(X) = p$ であることに注意して

$$
\begin{align}
V(X) &= \sum_{x=0,1} (x-\bar{x})^2 \Pr(X=x) \\
     &= (0-p)^2 (1-p) + (1-p)^2 p \\
     &= p^2(1-p) + (1-p)^2 p \\
     &= p^2 q + q^2 p  &\because q = 1 - p \\
     &= pq(p+q) \\
     &= pq             &\because p + q = 1
\end{align}
$$

### ダミー変数の平均値と標準偏差
ダミー変数とは，二値のカテゴリー変数（質的変数）を 1, 0の数値で表したもののことである。
たとえばアンケート調査で「現政権を支持するか」とか
「この1年間で海外旅行に行ったか」という質問に対する回答を
1または 0で表したデータがダミー変数である。

ここでいうダミーは「ニセモノ」とか「仮に設置したもの」という意味ではなく，
「代用物」という意味で，
**カテゴリーを数値で代用**したものくらいの意味である。
1 または 0 の二値で表すことにより，平均値により比率を得るといった，数値化が有用な場面で使用される手法である。

政権支持率調査で，100人中 70人が現政権支持ならば，
政権支持率は $\displaystyle\frac{70}{100} = 70\%$ であるが，
これをダミー変数で，現政権支持を 1, 不支持を 0で表せば，その平均値
$$
\frac{1\times 70 + 0\times 30}{100} = 70\%
$$
により政権支持率が得られる。

このことは，ベルヌーイ分布で言えば，
期待値 $E(X)$ が成功確率 $p$ に等しいことに対応する。
このことを応用すると標準偏差も計算できる。
分散は $pq$ だから標準偏差は $\sqrt{pq}$ である。

ダミー変数の標準偏差を求めるときは，$p, q$ を平均値 $\bar{x}$ を使って
$$
\begin{align}
p = \bar{x} \\
q = 1-p = 1-\bar{x}
\end{align}
$$
で置換える（substitute = 代入する）。
$$
S.D. = \sqrt{\bar{x}(1-\bar{x})}
$$
となる。

## ポアソン分布

二項分布 $B(n, p)$ に対し，
パラメータ $\mu = np$ を介して
$n\to\infty, p\to 0$ として得られる分布

$$
\mathrm{Pois}(\mu) := \lim_{n\to\infty} B\left(n, \frac{\mu}{n}\right)
$$

を考えてみたい。これは，多数 ($n\to\infty$) の確率的現象があるが，
まれにしか生じない ($p\to 0$) ため，
限られた頻度 ($\mu = np$) でしか観察されない事象の回数のモデルとして有用である。



$n$ を極端に大きく，$p$ を極端に小さくした分布を考えてみたい。


歴史的には，18世紀の数学者ド・モアブルや 19世紀の数学者ポアソンによる考察にさかのぼる。

この確率分布は**ポアソン分布** (Poisson Distribution) と呼ばれている。

を



実際の回数 ($n$) や確率 ($p$) が不明でも，
平均の発生頻度 $\mu$ がわかると，
一定期間内の発生回数をポアソン分布で見積もることができる。

たとえばA県では1年間に100名の人が交通事故で死亡しているとする。
死亡事故は3日に1度，均等に発生するわけでなく，
何日も事故が発生しなかったと思うと，1日に2件，3件のと死亡事故が発生することもある。
1日あたりの件数がポアソン分布で近似されるということである。

### 歴史

ポアソン分布の名称は，19世紀の数学者ポアソン (Siméon Poisson) にちなむ。
ポアソンは，不当な判決の件数を考察するにあたって，この分布を定式化した。

また統計学者ボルトケヴィッチは，
プロイセン陸軍で馬に蹴られて死亡した兵士数について調査し，
これが $\lambda = 0.61$ のポアソン分布でよく近似されることをしめした。

ただし先立つこと 100年以上前，
ド・モアブルは，スターリングの公式 ($n!$ の近似公式）の先行研究として
階乗や二項係数の見積もりについて調べており，
二項分布の極限として，こんにちでいうところの，
正規分布やポアソン分布を得ている。

Wolfram Language. (2007). PoissonDistribution. Wolfram Language & System Documentation Center. Retrieved from https://reference.wolfram.com/language/ref/PoissonDistribution.html

### 期待値と分散

ポアソン分布の期待値や分散は，二項分布の期待値，分散から得られる。すなわち

$$
\begin{align}
E(B(n, p)) &= np \\
V(B(n, p)) &= np(1-p)
\end{align}
$$

なので $p = \displaystyle\frac{\mu}{n}$ として

$$
\begin{align}
E(\mathrm{Pois}(\mu)) &= \lim_{n\to\infty} np = \frac{n\mu}{n} = \mu \\
V(\mathrm{Pois}(\mu)) &= \lim_{n\to\infty} np(1-p) = \lim_{p\to 0} \frac{\mu}{p}p(1-p) = \mu
\end{align}
$$

### 確率質量関数

ポアソン分布の確率質量関数 $\mathrm{pmf}(k) = \Pr(X = k)$ も
二項分布 $B(n, p)$ の確率質量関数を $p = \mu/n$ の下で $n\to\infty$ として導出できる。

$$
\begin{align}
\Pr(X = k) &= \lim_{n\to\infty} \frac{n!}{(n-k)!\, k!} p^k (1-p)^{n-k} \\
           &= \lim_{n\to\infty} \frac{n!}{(n-k)!\, k!}
                                \left(\frac{\mu}{n}\right)^k
                                \left(1-\frac{\mu}{n}\right)^{n-k} \\
           &= \frac{\mu^k}{k!}
              \lim_{n\to\infty} \frac{n!}{(n-k)!\,n^k}
                                \left(1-\frac{\mu}{n}\right)^n
                                \left(1-\frac{\mu}{n}\right)^{-k} \\
           &= \frac{\mu^k\,e^{-\mu}}{k!}
\end{align}
$$

なぜなら

$$
\frac{n!}{(n-k)!\,n^k} = \frac{\overbrace{n(n-1)(n-2)\cdots(n-k+1)}^k}
                               {\underbrace{n\cdot n\cdot\cdots\cdot n}_k}
                         \longrightarrow 1
$$

次に 

$$
\lim\left(1+\frac{1}{n}\right)^n = e
$$

より

$$
\lim\left(1-\frac{\mu}{n}\right)^n = e^{-\mu}
$$

最後は

$$
\lim\left(1-\frac{\mu}{n}\right)^k = 1
$$

### 使用法

`scipy.stats` モジュールの `poisson` によってポアソン分布を扱うことができる。
パラメータ &mu; はキーワード `mu` で与えることができる。

In [ ]:
from scipy.stats import poisson

μ = 0.5

dist = poisson(μ)        # 平均値 μ のポアソン分布オブジェクト
dist = poisson(mu = μ)   # 平均値をキーワード・パラメータ mu で明示してもよい

# 確率密度関数を f(x) で得るには
f = lambda x: dist.pmf(x)            # 確率分布オブジェクト dist から
f = lambda x: poisson.pmf(x, mu = μ) # 平均値パラメータを与えれば直接書くのも可能

A県の交通事故による死亡件数について，ポアソン分布でシミュレーションを行いたい。

In [ ]:
from scipy.stats import poisson
import pandas as pd

xs = range(5)
ys = poisson.pmf(xs, mu = 0.5)
pd.DataFrame({'確率': ys})

グラフを描いてみる。

In [ ]:
from scipy.stats import poisson
import matplotlib.pyplot as plt

xs = range(5)
ys = poisson.pmf(xs, mu = 0.5)
plt.bar(xs, ys)

ポアソン分布は二項分布の極限とのことなので，
$n$ を大きくし，それに合わせて $p$ を小さくすれば，
二項分布でも同様の結果が得られるはずだ。
試しに $n$ を A県の人口 1,000,000人，
$p$ を県民1人1人が1日に交通事故で死亡する確率 $\frac{0.5}{n}$ としよう。

In [ ]:
from scipy.stats import binom
import pandas as pd

n = 1_000_000
p = 0.5 / n
xs = range(n + 1)
ys = binom.pmf(xs, n, p)
df = pd.DataFrame({'確率': ys})
df.head(5)

この範囲では，ポアソン分布と同じ結果が得られている。
それより先は，微小なので無視してよいだろう。

以下では，ポアソン分布のグラフ（確率質量関数）を見ていく。

In [ ]:
from scipy.stats import poisson
import matplotlib.pyplot as plt

n = 5
xs = range(0, n+1)
ys = poisson.pmf(xs, mu = 1.2)
plt.bar(xs, ys)

峰 (peak) が $X = 1$ に移った。

パラメータ  &mu; をさらに大きくすると峰も右に移る。

In [ ]:
from scipy.stats import poisson
import matplotlib.pyplot as plt

n = 8
xs = range(0, n+1)
ys = poisson.pmf(xs, mu = 2.7)
plt.bar(xs, ys)

平均値 $\mu$ が大きくなるにつれて正規分布に近づく
（左右対称になってくる）。

In [ ]:
from scipy.stats import poisson
import matplotlib.pyplot as plt

n = 20
xs = range(n+1)
ys = poisson.pmf(xs, mu = 7.5)
plt.bar(xs, ys)

## 幾何分布

幾何分布とは，
成功確率 $p$ のベルヌーイ試行が成功するまでの回数の確率分布である。
たとえば

* コイントスで言えば，最初に表（おもて）になるまでコインを何回投げるか
* ボウリングあるいは野球の投球で言えば，確率 $p$ でストライクを取れる人が，最初にストライクをとれるまでの回数

`scipy.stats` の `geom` により幾何分布を扱うことができる。
パラメータは成功確率 $p$ である。

### 確率質量関数

**幾何分布**の名称は，確率質量関数が等比数列＝幾何数列，
累積分布関数が等比級数＝幾何級数になることに由来する。
確率質量関数は次のとおり

$$
\mathrm{pmf}(k) = \Pr(X = k) = (1-p)^{k-1} p.
$$

たとえば成功確率 $p=1/2$ の幾何分布の確率質量関数 PMF は

$$
\Pr(X = k) = \left(\frac{1}{2}\right)^{k-1}\left(\frac{1}{2}\right)
           = \left(\frac{1}{2}\right)^k
$$

In [ ]:
from scipy.stats import geom
import pandas as pd

p = 0.5
dist = geom(p)
xs = range(5)
ys = dist.pmf(xs)
pd.DataFrame({'確率': ys})

$X = 0$ の確率が 0なのが気になるかもしれない。
最初から成功すると試行回数は 1, すなわち $X = 1$ なので
確率変数の範囲は $X \ge 1$ である。
$X = 0$ の確率は 0として扱われる ($\Pr(X = 0) = 0$)。

そこで $X$ の範囲を 1以上にするとともに，
pandas のデータフレームの index もそれに合わせる。

In [ ]:
from scipy.tats import geom
import pandas as pd

p = 0.5
dist = geom(p)
xs = range(1, 6)
ys = dist.pmf(xs)
pd.DataFrame({'確率': ys}, index = xs)

In [ ]:
import matplotlib.pyplot as plt

plt.bar(xs, ys)

### 累積分布関数

累積分布関数 $\mathrm{cdf}(n)$ は確率質量関数 $\mathrm{pmf}(k)$ の累積である。

$$
\mathrm{cdf}(n) = \Pr(X\le n) = \sum_{k=1}^n \Pr(X=k)
                = 1 - (1-p)^n
$$

$q = 1-p$ として

$$
\begin{align}
\mathrm{cdf}(n) &= \sum_{k=1}^n (1-p)^{k-1}p \\
                &= \sum_{k=1}^n q^{k-1}p \\
                &= (1 + q + q^2 + \cdots + q^{n-1})p \\
                &= \frac{1-q^n}{1-q}\cdot p \\
                &= \frac{(1-q^n)p}{p} \\
                &= 1-q^n \\
                &= 1 - (1-p)^n
\end{align}
$$

In [ ]:
from scipy.stats import geom
import pandas as pd

p = 0.5
dist = geom(p)
xs = range(1, 6)
ys = dist.cdf(xs)
pd.DataFrame({'累積確率': ys}, index = xs)

### 生存関数

$\mathrm{sf}(n) = 1 - \mathrm{cdf}(n)$ なので更に簡単な式になる。

$$
\mathrm{sf}(n) = \Pr(X > n) = 1 - \mathrm{cdf}(n)
                = (1-p)^n = q^n
$$

$q$ は失敗確率 $q = 1-p$ のことであった。

In [ ]:
from scipy.stats import geom
import pandas as pd

p = 0.5
dist = geom(p)
xs = range(1, 6)
ys = dist.sf(xs)
pd.DataFrame({'生存確率': ys}, index = xs)

幾何分布の
パラメータ $p$ ごとの確率質量関数のグラフの違いを見てみよう。

In [ ]:
from scipy.stats import geom
import numpy as np
import matplotlib.pyplot as plt

xs = np.arange(1, 7)

ys6 = geom.pmf(xs, 0.6)
plt.bar(xs-0.25, ys6, width=0.25)

ys5 = geom.pmf(xs, 0.5)
plt.bar(xs, ys5, width=0.25)

ys4 = geom.pmf(xs, 0.4)
plt.bar(xs+0.25, ys4, width=0.25)

plt.legend(["p=0.6", "p=0.5", "p=0.4"])

成功確率 $p$ が大きいほど
すぐに成功する（＝寿命が短い）ことが見て取れる。

生存関数のグラフも見てみる。

In [ ]:
from scipy.stats import geom
import numpy as np
import matplotlib.pyplot as plt

xs = np.arange(1, 7)

ys6 = geom.sf(xs, 0.6)
plt.bar(xs-0.25, ys6, width=0.25)

ys5 = geom.sf(xs, 0.5)
plt.bar(xs, ys5, width=0.25)

ys4 = geom.sf(xs, 0.4)
plt.bar(xs+0.25, ys4, width=0.25)

plt.legend(["p=0.6 (q=0.4)", "p=0.5 (q=0.5)", "p=0.4 (q=0.6)"])

失敗確率が高いほど寿命が長い，
同じことだが，成功確率が高いほど試行回数が少ないことが見て取れる。

### 別の定義

幾何分布を「成功するまでの失敗回数」と定義する流儀があるので混乱しないでもらいたい。
この定義では，初回で成功するのは，1度も失敗しないということなので，
確率変数 $X = 0$ となる。確率変数の範囲は $X \ge 0$ である。
確率質量関数は 
$$
\Pr(X = k) = q^k p
$$
と少し簡単な式になる。ただし $q$ は失敗確率 $q = 1 - p$

累積分布関数は
$$
\Pr(X \le n) = \sum_{k=0}^n q^k p = (1 + q + \cdots + q^n)p = 1 - q^{n+1}
$$
生存関数は
$$
\Pr(X > n) = 1 - \mathrm{cdf}(n) = q^{n+1}
$$

`scipy.stats.geom` でも，位置パラメータ `loc = -1` を与えると，
この定義の幾何分布を扱える。

In [ ]:
from scipy.stats import geom
import numpy as np
import matplotlib.pyplot as plt

xs = np.arange(7)
ys = geom.pmf(xs, 0.5, loc = -1)
pd.DataFrame({'確率': ys})

### 幾何分布のモーメント（期待値など）

幾何分布 $G(p)$ の
平均値（期待値），分散，歪度，尖度は，それぞれ次のとおり。
ただし $q = 1-p$ つまり失敗確率とする。

$$
\begin{align}
E(X) &= 1/p \\
\mathrm{Var}(X) &= \frac{1-p}{p^2} = \frac{q}{p^2} \\
\mathrm{Skew}(X) &= \frac{2-p}{\sqrt{1-p}}
    = \frac{q+1}{\sqrt{q}} = \sqrt{q} + \frac{1}{\sqrt{q}} \\
\mathrm{Kurt}(X) &= \frac{p^2-6p+6}{1-p} = 4 + q + \frac{1}{q}
\end{align}
$$

## 指数分布

指数分布 (Exponential Distribution) は，幾何分布の連続版と言われる。
次のように替えたものを考えればよい。

| 幾何分布（離散） | 指数分布（連続） |
|---------------|----------------|
| 試行回数 $k$    | 時間 $t$       |
| 1回の試行での成功確率 | 一定の時間における成功確率 |
| 成功するまでの試行回数 | 成功するまでの時間 |

幾何分布における 1回ごとの試行は，
非復元抽出の独立反復試行，
すなわち，前の試行の結果の影響を受けず，毎回同じ成功確率の試行であることにあることも注意したい。
これは指数分布においては，過去の事象の影響を受けず，
常に一定の割合で起きる確率的現象となろう。

### 具体例

典型的な例が放射性同位体の寿命である。
放射性同位体の原子核は，他の原子と無関係に一定の確率で崩壊して，別の物質に変わる。
放射性同位体の原子1つ1つの寿命は，指数分布に従う。

医薬品など体内に入った物質は，肝臓等で分解されたり，尿として排泄されて減っていく。
減っていく割合は個人（個体）ごと異なるとしても，
同一個人（同一個体）ならさほど変わらないだろうから，
体内に残る時間は指数分布に従うと考えてよいだろう。

塗装やコーティングは，
日光にあたったり，空気中の酸素と反応することで劣化していく。
劣化速度は安定していると仮定できることも多いだろう。
その場合，寿命のモデルとして指数分布を使うことができる。

時期に関わらず一定の割合で売れる商品は，
在庫として残る期間を指数分布でモデル化できる。
ただし，この仮定が成り立つ商品は限られるだろう。
たとえば野菜は新鮮さに価値があって，時間が経つと売れにくくなるだろう。
季節によっても売れ行きが変動する。

### 指数分布のグラフ

指数分布の確率変数は生存期間だが，
多数の粒子を思い浮かべ，その寿命の分布を想起せよと言われても，
容易に感じる人は少ないと思う。
指数分布する現象は，生存数の減少の観点で捉えた方がわかりやすいだろう。

どちらも似た形状のグラフになるが，
* 確率密度関数は寿命の分布を
* 生存関数は寿命を迎えていない個体の割合
を表す。グラフを見れば把握できる。

`scipy.stats` では指数分布を `expon` という名前で提供している。
連続確率分布では，確率質量関数 (Probability Mass Function) ではなく
確率密度関数 (Probability Density Function) によって確率の過多を表す。

In [ ]:
from scipy.stats import expon
import numpy as np
import matplotlib.pyplot as plt

xs = np.linspace(0, 4.5)
ys = expon.pdf(xs)
plt.title('Probability Density Function')
plt.ylim(0, 1.1)
plt.plot(xs, ys)

In [ ]:
from scipy.stats import expon
import numpy as np
import matplotlib.pyplot as plt

xs = np.linspace(0, 4.5)
ys = expon.sf(xs)
plt.title('Survival Function')
plt.ylim(0, 1.1)
plt.plot(xs, ys)

確率や統計の教科書で指数分布は，
パラメータ &lambda; を使って $\mathrm{Exp}(\lambda)$ のように表すことが多い。
指数分布にパラメータを与えるのに `scipy.stats` では
キーワード引数 `scale` に $1/\lambda$ を与える。

In [ ]:
from scipy.stats import expon
import numpy as np
import matplotlib.pyplot as plt

xs = np.linspace(0, 4.5)

ys1 = expon.pdf(xs, scale = 1)
plt.plot(xs, ys1)
ys2 = expon.pdf(xs, scale = 1 / 0.7)
plt.plot(xs, ys2)
ys3 = expon.pdf(xs, scale = 1 / 0.5)
plt.plot(xs, ys3)

plt.legend(["λ=1.0", "λ=0.7", "λ=0.5"])

パラメータ &lambda; は，確率密度関数のグラフでいうと
$x = 0$ のときの高さを表している。



幾何分布を表す関数は，確率質量関数よりも累積分布関数が，
それよりも生存関数が簡単な式で表すことができた。
同じく，指数分布も生存関数が最も簡単な式になる。

指数分布の生存関数は，生存している確率が
$$
a^{-x}
$$
で表される確率分布である。ただし $a>1$ とする。
累積分布関数は
$$
1-a^{-x}
$$
である。
確率密度関数は，これを微分したものである。
計算を簡単にするため，補助となるパラメータ $\lambda$（ラムダ）を
$$
\lambda = \log a
$$
と定める。対数関数は自然関数の逆関数だから
$$
e^{\lambda} = a
$$
となる。これを利用して累積分布関数を微分する。
$$
\begin{align}
(1-a^{-x})' &= ((1 -(e^{\lambda})^{-x})' \\
            &= (1-e^{-\lambda x})' \\
            &= -(-\lambda e^{-\lambda x}) \\
            &= \lambda e^{-\lambda x}
\end{align}
$$
ここからさらに変形して $\lambda a^{-x} = a^{-x}\log a$ を得ることは可能だが，
パラメータ $\lambda$ をそのまま使って
$$
\mathrm{pdf}(x) = \lambda e^{-\lambda x}
$$
とする習慣である。

指数分布は，このパラメータ $\lambda$ を使って $\mathrm{Exp(\lambda)}$ と表記する。

確率や統計の教科書を見ても，パラメータ $\lambda$ の意味が書かれていないことが多いが，
これは
$$
a^{-x} = e^{-\lambda x}
$$
となる $\lambda$ のことである。$\lambda$ は減衰速度と$x = -1$ を代入すれば
$$
a = e^\lambda
$$
であり，対数を取ると
$$
\lambda = \log a
$$
である。

### パラメータと期待値

パラメータ <ruby>&lambda;<rp>（</rp><rt>ラムダ</rt><rp>）</rp></ruby> の
意味は期待値等を見るとわかる。
次は，期待値，分散，歪度，尖度である。
$$
\begin{align}
E(X) &= \frac{1}{\lambda} \\
\mathrm{Var}(X) &= \frac{1}{\lambda} \\
\mathrm{Skew}(X) &= 2 \\
\mathrm{Kurt}(X) &= 6 \\
\end{align}
$$

左右対称な分布の歪度は 0である。2は，大きく左に偏っていることを意味する。
尖度は分布の裾の重さを表していて， 6というのは，他では見ない大きさである。

### 確率密度関数
$$
\mathrm{pdf}(x) = \lambda\exp(-\lambda x)
$$

In [ ]:
from scipy.stats import expon
import numpy as np
import matplotlib.pyplot as plt

λ = 1.0
dist = expon(scale = λ)
xs = np.linspace(0, 4, 100)
ys = dist.pdf(xs)
plt.ylim(0.0, 1.0)
plt.plot(xs, ys)

In [ ]:
λ = 2.0
ys = expon.pdf(xs, scale = λ)
plt.ylim(0.0, 1.0)
plt.plot(xs, ys)

### 累積分布関数と生存関数

確率密度関数 (pdf) を定積分すると累積分布関数 (cdf) になる。

$$
\mathrm{cdf}(x) = 1 - \exp(-\lambda x)
$$

In [ ]:
ys = dist.cdf(xs)
plt.ylim(0.0, 1.0)
plt.plot(xs, ys)

生存関数は

$$
\mathrm{sf}(x) = 1 - \mathrm{cdf}(x) = \exp(-\lambda x)
$$

$a = e^\lambda$ となるように $a$ を定めれば

$$
\mathrm{sf}(x) = a^{-x}
$$

逆に言うと，指数分布とは生存関数が $a^{-x}$ である確率分布である。
この $a$ を用いると，累積分布関数は
$$
\mathrm{cdf}(x) = 1 - a^{-x}
$$
確率密度関数は
$$
\mathrm{pdf}(x) = \lambda a^{-x} = a^{-x}\log a
$$
となる。確率密度関数に現れる係数 $\lambda = \log a$ は，全事象の確率を1とする
$$
\int_{-\infty}^{\infty} \lambda a^{-x}\,dx = 1
$$
ための係数ととらえることもできる。

In [ ]:
ys = dist.sf(xs)
plt.ylim(0.0, 1.0)
plt.plot(xs, ys)

## 負の二項分布

成功確率 $p$ のベルヌーイ試行を $n$ 回成功するまでに
失敗した回数の分布を負の二項分布という。

失敗回数 $X$ に対し $Y = X + 1$ とすると $Y$ は成功するまでの試行回数なので，
$n = 1$ のとき $Y$ の分布は**幾何分布**である。
つまり，幾何分布は負の二項分布の特別な例である。

負の二項分布の名称は，
確率質量関数が**負の二項係数**になることに由来する。
コイントスの確率分布とはおよそ関係ないので混乱しないでもらいたい。
二項係数との関係はいったん無視して，どういう分布になるか見ていく。

In [ ]:
from scipy.stats import nbinom
import pandas as pd

p = 0.5
n = 3
dist = nbinom(n, p)
xs = range(15)
ys = dist.pmf(xs)
pd.DataFrame({'割合': ys})

In [ ]:
import matplotlib.pyplot as plt

plt.bar(xs, ys)

負の二項分布に関しては

* 何回で定着するか
* 何回で合格するか

が関心事であることが多いだろう。
これは累積分布関数で得られる。

In [ ]:
ys = dist.cdf(xs)
pd.DataFrame({'割合': ys})

累積分布関数から次のことが読み取れる。
* $X=2$ で 50%; 失敗回数 2, 成功回数 _n_ = 3, 計試行回数5回で 50% の定着
* $X=7$ で 95%; 失敗回数 7, 成功回数 _n_ = 3, 計試行回数 10回で 95% の定着

In [ ]:
import matplotlib.pyplot as plt

plt.bar(xs, ys)

### 確率質量関数の導出

$n$ 回成功するまでに $k$ 回失敗する事象について調べる。
具体的に「3回成功するまでに2回失敗する」組合せを列挙する。
`O` を成功，`X` を失敗とする。

| 事象 |
|-----|
|XXOOO|
|XOXOO|
|XOOXO|
|OXXOO|
|OXOXO|
|OOXXO|

5回目の試行では必ず成功するので，
* 4回の試行で2回成功する組合せ
* （同じことだが）4回の試行で2回失敗する組合せ

を考えればよい。その組合せの個数は
$$
{}_4\mathrm{C}_2 = \frac{4!}{2!\cdot 2!} = 6
$$
である。

一般化する。次の組合せ（事象）は同一である。
* $n$ 回成功するまでに $k$ 回失敗
* $k + n$ 回試行し，$k$ 回失敗，最後の1回は成功
* $k + n - 1$ 回試行し $k$ 回失敗
* $k + n - 1$ 回試行し $n-1$ 回成功

組合せの個数は
$$
{}_{k + n - 1}\mathrm{C}_k = {}_{k + n - 1}\mathrm{C}_{n-1} =
    \frac{(k+n-1)!}{k!\cdot (n-1)!}
$$

成功確率 $p$, 失敗確率 $q = 1-p$ で，
$n$ 回成功する負の二項分布の確率変数 $K$ の確率密度関数は，
$K$ 回失敗する確率であり

$$
f(K) = p^n q^K \frac{(K + n  - 1)!}{K!\cdot(n-1)!}
$$

### SymPy の利用

負の二項係数について知るため SymPy を使いたい。

`math` モジュールの `comb` 関数を使うと二項係数を計算できるのだが，
この関数は負の数に対応していない（負の数を与えるとエラーになる）。
SymPy という記号計算 (symbolic computation) のモジュールに
`binomial` という二項係数を計算する関数があり，これが負の数に対応している。

日常的に Python を使っている利用者には
「SymPy をインストールせよ」の一言で解決する問題だが，
そうでないと何をしたらよいかわからないし，
さらには「落とし穴」まであるせいで，
一概にこうせよと指示することができない。
少々長くなるが，モジュールのインストール方法を述べる。

なおインストールしないという選択肢もある。
このノートブックで SymPy を使うのは，
次のセクションだけである。
**負の二項分布**自体は `scipy.stats.nbinom` にあり SymPy を必要としない。
セルの先頭に

```jupyter
import sympy
```

とか

```jupyter
from sympy import ...
```

と書かれているセルをスキップすれば，
SymPy をインストールしなくてもこのノートブックを使える。

ではモジュールのインストールに方法を説明する。
まず，使用している Python が
`PIP` と `conda` のどちらのパッケージ・システムで
管理されているかを把握する必要がある。
`conda` がインストールされていれば `conda` で管理しているだろうし，
インストールされていなければ `PIP` で管理しているであろう。
Jupyter の「ターミナル」を開いて次を実行する。

```sh
conda list
```

あるいは Jupyter のセルで次を実行する。

```ipython
%conda list
```

`conda` がインストールされていれば，インストール済みモジュール一覧が表示される。
なお `conda` で管理していても `PIP` でのインストールは可能で，
`conda` で提供していないモジュールをインストールするのに `PIP` を使うことがある。
しかし一般には，両者を混ぜて使うと，管理しているバージョン番号に不整合が生じて，
後々問題が起きる可能性が高い。一方でだけ使うべきである。

In [ ]:
%conda list

つづいて SymPy をインストールする。

`conda` でパッケージ管理されているなら
Jupyter のターミナルで次を実行する。

```sh
conda install sympy
```

ターミナルの開き方がわからなければ Jupyter のセルで次を実行する。
```jupyter
%conda install sympy
```

一方 `PIP` で管理しているならターミナルで次を実行する。
```sh
python -m pip upgrade pip
python -m pip install sympy
```

日常的に Python を使っていれば，
`python -m pip` の部分を `pip` で置き換えてよいだとか，
いや `pip3` にするだとか，
Jupyter のセル内で `%pip install sympy` でもできるだとか知っているだろう。
自己責任でやってかまわない（落とし穴はあるのであくまでも自己責任で）。

### 二項係数から負の二項係数へ

`math` モジュールの `comb` 関数を使うと
二項係数を計算できる。
ここからパスカルの三角形を作れる。

In [ ]:
from math import comb

for n in range(10):
    print([ comb(n, k) for k in range(0, n+1) ])

中央寄せ (centering) した方が見やすいかもしれない。
二項係数は，左上と右上を加算した値になっていることが確認できる。

In [ ]:
from math import comb

for n in range(10):
    print(str([ comb(n, k) for k in range(0, n+1) ]).center(50))

$k > n$ のとき ${}_n\mathrm{C}_k　= 0$ である

In [ ]:
from math import comb

for n in range(10):
    print([ comb(n, k) for k in range(0, 11) ])

これについても，左上と真上を加算することで
二項係数が得られることが確認できる。

これを $n < 0$ にも拡張するというのが
これからやりたいことである。

`sympy.binomial` を使うと $n < 0$ や $k < 0$ も含めて二項係数を計算できる。
まず $n \geq 0$ だが $k < 0$ のときどうなるか試してみる。

In [ ]:
from sympy import binomial

for n in range(10):
    print([ binomial(n, k) for k in range(-2, 10) ])

確かに，左上と真上を加算した値になっている。

続いて $n < 0$ も調べてみる。

In [ ]:
from sympy import binomial
# from sympy import symbols, var

for n in range(-9, 10):
    print([ binomial(n, k) for k in range(-2, 10) ])

左上と真上を加算した値になっていることを確認してもらいたい。

$n < 0$ のときの二項係数は次の計算を行なっている。

$m = -n$ とすると
$$
(-1)^k\binom{-m}{k} = 
  \left|\binom{-m}{k}\right| = 
  {}_m\mathrm{H}_k =
  \binom{k + m - 1}{m - 1} =
  \binom{k + m - 1}{k}
$$

${}_m\mathrm{H}_k$ は**重複組合せ**（後述）の数で，
$m$ 種類から重複を許して $k$ 個選ぶ組み合わせの数を表す。

${}_m\mathrm{H}_k = \displaystyle\binom{k + m - 1}{k}$  を
`math.comb` で計算して確かめる。

In [ ]:
from math import comb

for m in range(1, 10):
    print([ comb(k + m - 1, k) for k in range(10) ])

### 重複組合せとは

例題：
赤，白，黒の3種類の玉が抽選箱に入っているとき
4個取り出すときの組み合わせの数はいくらか。

解答：赤，白，黒の順になるようにソーティングして列挙する。

| 玉の組み合わせ | 仕切りの位置 |
|----------|------------|
| 赤赤赤赤 | `....++` |
| 赤赤赤白 | `...+.+` |
| 赤赤赤黒 | `...++.` |
| 赤赤白白 | `..+..+` |
| 赤赤白黒 | `..+.+.` |
| 赤赤黒黒 | `..++..` |
| 赤白白白 | `.+...+` |
| 赤白白黒 | `.+..+.` |
| 赤白黒黒 | `.+.+..` |
| 赤黒黒黒 | `.++...` |
| 白白白白 | `+....+` |
| 白白白黒 | `+...+.` |
| 白白黒黒 | `+..+..` |
| 白黒黒黒 | `+.+...` |
| 黒黒黒黒 | `++....` |

このように玉の組み合わせは 15通りある。
これは玉と仕切りを併せたもの（6個）から仕切りを2個選ぶ組合せと同じである。

$$
{}_3\mathrm{H}_4 = {}_6\mathrm{C}_2 = \frac{6!}{4!\cdot 2!} = \frac{6\cdot 5}{2} = 15
$$

一般化すると，
> $n$ 種類の玉から $k$ 個選ぶときの重複組合せ

は，玉が $k$ 個，仕切りが $n-1$ 個のとき，

> 玉と仕切りを併せたもの $k + n - 1$ 個から，仕切り $n-1$ 個を選ぶ，
> あるいは玉 $k$ 個を選ぶ組合せ

と対応する。その個数は

$$
{}_n\mathrm{H}_k = {}_{k + n - 1}\mathrm{C}_{n - 1}
$$

負の二項分布の定義まで振り返ろう。
これは $n$ 回成功するまでに $k$ 回失敗する組合せの個数に等しい。

## 超幾何分布

**超幾何分布** (hypergeometric distribution)は
二項分布の非復元抽出バージョンである。

復元抽出では，抽選箱から取り出した玉は抽選箱に戻して，それから次の試行を行う。
すべての試行で成功確率は同一である。

非復元抽出では，抽選箱から取り出した玉を抽選箱に戻さずに次の試行を行う。
そのため
* 試行が成功すれば，当たりの球が減るので次の試行の成功確率が減る。
* 試行が失敗すれば，当たりの球が減らずに全体の球が減るので成功確率が高くなる。

超幾何分布は次のパラメータで定まる。

* 全体の個数 $M$
* 当たりの個数 $n$
* 試行回数 $N$

確率変数は，当たりが選ばれた個数である。

たとえばロシアンルーレットは，
全体の個数 $M=6$, 当たりの個数 $n = 1$ の超幾何分布とみなせる。
試行回数 $N=6$ とすると

確率変数 $K=0$ となる確率は

$$
\frac{5}{6}\cdot\frac{4}{5}
           \cdot\frac{3}{4}
           \cdot\frac{2}{3}
           \cdot\frac{1}{2}
           \cdot\frac{0}{1} = 0
$$

である。 $K=1$ となる確率は

$$
\frac{1}{6}+\frac{5}{6}\cdot\frac{1}{5}
           +\frac{5}{6}\cdot\frac{4}{5}\cdot\frac{1}{4}
           +\cdots
           +\frac{5}{6}\cdots\frac{1}{2}\cdot\frac{1}{1}
           = 1
$$

In [ ]:
from scipy.stats import hypergeom
import pandas as pd

trials = 6
dist = hypergeom(6, 1, trials)
xs = range(2)
ys = dist.pmf(xs)
pd.DataFrame({'割合': ys})

このように，事象ごとに確率を計算すると，
試行するごとに玉の数が減り，
分母が ${}_n\mathrm{P}_r$ で表される式になる。
この式が**超幾何数列**と呼ばれるものに分類されることから，
この確率分布は**超幾何分布**と呼ばれている。
幾何分布とは名前が似ていても関係ない。

超幾何数列の定義に踏み込むと大変なので，基本イメージを紹介する。
無限級数とかテーラー展開を知っていることが前提である。

三角関数，指数関数を無限級数展開すると係数の分母に階乗が現れる。
他の関数でも，しばしば係数の分母，分子に，階乗や階乗を拡張したような式が現れる。
このような級数を，（等比級数＝幾何級数よりも複雑だという意味で）超幾何級数と
名前を付けて分類した。
数学の公式集があれば，無限級数展開の係数を眺めてもらいたい。
多くは超幾何級数である。

超幾何級数を微分しても超幾何級数になることから，
微分方程式の解法として研究された。

$$
\mathrm{pmf}(K) = \frac{({}_n\mathrm{C}_K)({}_{M-n}\mathrm{C}_{N-K})}
                       {{}_M\mathrm{C}_N}
$$

## 一様分布と三角分布

### 離散型一様分布

`randint(low, high)` で `range(low, high)` の範囲
すなわち `low` 以上 `high` 未満の離散型一様分布が得られる。

In [ ]:
from scipy.stats import randint
import matplotlib.pyplot as plt

low = 0
high = 7
xs = [ x for x in range(low, high) ]
ys = randint.pmf(xs, low, high)
plt.bar(xs, ys)

`randint(low, high)` のモーメント統計量は次のとおり。
ただし $n$ はビンの個数すなわち high &minus; low

| 統計量 | 値 |
|-------|----|
| 平均値 | (low + high - 1)/2 |
| 分散  | $(n^2 - 1)/12$ |
| 歪度 | 0 |


In [ ]:
from scipy.stats import randint

mean, var, skew = randint(0, 7).stats(moments='mvs')
print('平均', mean)
print('分散', var)
print('歪度', skew)

### 連続型一様分布

> `uniform(loc, scale)`

で連続型の一様分布が得られる。
位置パラメータ `loc` は最小値，
スケールパラメータ `scale` は範囲の大きさである。
`loc` + `scale` が上限値となる。

逆に `left` 以上 `right` 未満の連続一様分布を得たいなら

> `uniform(loc = left, scale = right - left)`

とする。

In [ ]:
from scipy.stats import uniform
import numpy as np

left = 1
right = 7
dist = uniform(loc = left, scale = right - left)
xs = np.linspace(left-1.5, right+1.5, 600)
ys = dist.pdf(xs)
plt.fill_between(xs, ys, 0, alpha = 0.3, hatch = '//')
plt.plot(xs, ys)

`uniform(loc, scale)` のモーメント統計量は次のとおり。


| 統計量 | 値 |
|-------|----|
| 平均値 | loc + scale / 2 |
| 分散  | scale<sup>2</sup>/12 |
| 歪度 | 0 |
| 尖度 | &minus;6/5 |


In [ ]:
m, v, s, k = dist.stats(moments='mvsk')

print(m, v, s, k)

In [ ]:
dist.ppf([0/4, 1/4, 2/4, 3/4, 4/4])

### 三角分布

2つの一様分布の和は**三角分布**というものになる。
正確には
>（独立に）一様分布に従う 2つの確率変数 $X$, $Y$ の和 $X+Y$ は
> 三角分布に従う。

<!-- いわゆる独立同一分布 (iid &mdash; independent and ideintically distributed) より条件が緩い。$X$, $Y$ の確率分布が異なる場合，（$X + Y$ の確率密度関数は）左右非対称になる。 -->

にわかにピンとくる命題ではないが，
手がかりさえつかめれば難しい話ではない。
2つの6面サイコロを振ったときのサイコロの目の合計について，事象を列挙してみよう。

In [ ]:
import pandas as pd

freq = pd.DataFrame(0,
                    columns = [ i + 1 for i in range(6)],
                    index   = [ i + 1 for i in range(6)])
for i in freq.columns:
    for j in freq.index:
        freq.loc[i, j] = i + j

display(freq)

サイコロを振ってどの目になるかは一様分布になるであろうから，サイコロの目の合計に着目して事象を数え上げれば，その分布がわかる。

In [ ]:
# 前のセルを実行していることが前提
# 変数 freq の値を定めておかないといけない

import numpy as np

# freq の集計表
hist = np.histogram(freq, bins=11)     # bins = len([2, 3, ..., 12]

# 集計表を棒グラフにする
import matplotlib.pyplot as plt
ys = hist[0]
xs = range(len(ys))
plt.bar(xs, ys, tick_label = [x + 2 for x in xs])

このように三角形の分布になる。
連続一様分布の和であれば，三角分布 `triang` で得られる。

`triang(c, loc, scale)` は，
左端が `loc`, 底辺の長さが `scale`, 
頂点の位置が `scale` のどの割合かを示す `c` で定まる三角分布である。
$$
0.0 < c < 1.0
$$
`loc` と `scale` のデフォルト値は，それぞれ 0.0 と 1.0 である。

たとえば `triang(0.5, loc = 0, scale = 2)` は，
独立に `uniform(0, 1)` に従う 2つの確率変数の和の確率分布である。

In [ ]:
from scipy.stats import triang
import numpy as np
import matplotlib.pyplot as plt

xs = np.linspace(-0.5, 2.5, 101)
ys = triang.pdf(xs, c=0.5, loc = 0, scale = 2)
plt.plot(xs, ys)

三角分布は単純な仕組みの確率分布であり簡単に作れる。性質もわかりやすいので，単峰かつ $X$ の範囲が有限な確率分布の例として有用である。